In [ ]:
import sys
from pathlib import Path

# Walk upward from the current working directory until we find the
# repository root. We define the repo root as the folder that contains
# both `src/` (our Python package) and `data/` (our datasets).
#
# This is necessary because Jupyter in VS Code / Codespaces often runs
# with cwd = /.../notebooks instead of the project root.

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "data").exists():
            return p
    raise RuntimeError(f"Could not find repo root above {start}")

# Determine the true repository root regardless of where the notebook
# kernel was launched from

REPO_ROOT = find_repo_root(Path.cwd())

# Add the repo root to Python's import search path so that
# `import src.io` works inside notebooks

sys.path.insert(0, str(REPO_ROOT))

# Build an absolute path to the raw data directory so we never rely on
# fragile relative paths like "data/raw"

RAW_DIR = REPO_ROOT / "data" / "raw"
RAW_DIR = RAW_DIR.resolve()

# Now that paths and imports are stable, we can safely import libraries
# and our project code

import pandas as pd
from src.io import ingest_raw_csvs

# Display for sanity checking
REPO_ROOT, RAW_DIR

In [ ]:
import os

# Show the actual current working directory of the notebook kernel.
# In VS Code / Codespaces this is usually /.../notebooks instead of the repo root.
print("cwd:", os.getcwd())

# Show the repository root we detected via the bootstrap logic.
# This should point to /workspaces/protected-bike-lanes-ridership
print("repo_root:", REPO_ROOT)

# Further sanity check that the src/ package is actually present at the repo root.
# If this is False, imports like `from src.io import ...` will fail.
print("src exists:", (REPO_ROOT / "src").exists())

# Further sanity check that the data/ directory is present.
# If this is False, RAW_DIR construction is broken.
print("data exists:", (REPO_ROOT / "data").exists())

# Show the first entry on Python's module search path.
# This should be the repo root, meaning Python can find `src/` as a package.
print("sys.path[0]:", sys.path[0])

In [ ]:
# Run the raw CSV ingest pipeline against the raw data directory.
# This reads every CSV under data/raw/, applies any safe parsing logic,
# and combines them into a single pandas DataFrame.
result = ingest_raw_csvs(RAW_DIR)

# The unified dataframe of all counter data
df = result.df

# Basic sanity checks so we know we didn't silently fail
print("Rows:", len(df))                      # total number of records loaded
print("Columns:", len(df.columns))           # how wide the dataset is
print("Files read:", len(result.files_read)) # how many CSVs were successfully ingested
print("Files failed:", len(result.files_failed))  # how many CSVs could not be read

# Peek at the first few rows to visually confirm the structure
df.head()

In [ ]:
# List all column names in sorted order so we can see the full schema
# and spot things like multiple date fields, direction columns, or
# inconsistently named variables across files.
pd.Series(df.columns).sort_values()

In [ ]:
# Identify any columns that look like date or time fields.
# Seattle data often includes multiple timestamp columns, so this
# helps us see our candidates before deciding which one to use
# for daily aggregation.
[c for c in df.columns if "date" in c or "time" in c]

In [ ]:
# Compute the fraction of missing values in each column and show
# the 20 most incomplete fields. This tells us which columns are
# mostly empty metadata and which ones are reliable enough to use
# for analysis and modeli
df.isna().mean().sort_values(ascending=False).head(20)